In [2]:
import pandas as pd
import numpy as np
import re

import os
import datetime
import matplotlib.pyplot as plt

from copy import deepcopy
%matplotlib inline
plt.style.use('ggplot')

In [ ]:
# load the data

In [3]:
df = pd.read_csv("examen2.csv")
 
print("=" * 60)
print("ORIGINAL DTYPES")
print("=" * 60)
print(df.dtypes)
print("\nShape:", df.shape)

ORIGINAL DTYPES
ubicado_en_avenida                     str
empresa_recolectora_de_dinero          str
abierto_en_fines_de_semana          object
dias_para_recoleccion_de_dinero    float64
municipio                              str
cantidad_de_camaras                    str
banco_inseguro                       int64
cantidad_de_empleados                  str
dtype: object

Shape: (15147, 8)


In [5]:
df.head(10)

,ubicado_en_avenida,empresa_recolectora_de_dinero,abierto_en_fines_de_semana,dias_para_recoleccion_de_dinero,municipio,cantidad_de_camaras,banco_inseguro,cantidad_de_empleados
0,Falso,Touché-Coulé,NaN,7.0,zapopan,N.A.,0,hay 6.5 trabajadores para los dias habiles
1,NaN,Touché-Coulé,NaN,21.0,Guadalajara,N.A.,1,8 Trabajadores
2,Falso,Touché-Coulé,NaN,18.0,zapopan,N.A.,0,hay 10 trabajadores para los dias habiles
3,True,Touché-Coulé,False,12.0,zapopan,6.0,0,tenemos 8.5 trabajadores para los 7 dias de la...
4,NaN,Touché-Coulé,NaN,15.0,zapopan,N.A.,0,contamos con 159.5 trabajadores para los 5 dia...
5,True,Touché-Coulé,False,16.0,Guadalajara,5.0,1,12 trabajadores
6,Falso,Seguristark,NaN,25.0,Zapopan,N.A.,1,tenemos 11 trabajadores para los 7 dias de la ...
7,True,Seguristark,NaN,17.0,zapopan,N.A.,0,tenemos 10.5 trabajadores para los 7 dias de l...
8,True,Touché-Coulé,NaN,13.0,Guadalajara,N.A.,0,hay 7 trabajadores para los dias habiles
9,True,Touché-Coulé,NaN,17.0,Guadalajara,N.A.,0,12.5 Trabajadores


In [7]:
df.tail(10)

,ubicado_en_avenida,empresa_recolectora_de_dinero,abierto_en_fines_de_semana,dias_para_recoleccion_de_dinero,municipio,cantidad_de_camaras,banco_inseguro,cantidad_de_empleados
15137,Falso,Touché-Coulé,NaN,17.0,Guadalajara,N.A.,1,contamos con 9 trabajadores para los 5 dias ha...
15138,True,Touché-Coulé,NaN,18.0,Guadalajara,N.A.,0,9.5 Trabajadores
15139,Falso,Touché-Coulé,NaN,12.0,Zapopan,N.A.,1,contamos con 11.5 trabajadores para los 5 dias...
15140,Falso,Touché-Coulé,False,14.0,zapopan,N.A.,1,contamos con 11 trabajadores para los 5 dias h...
15141,True,Touché-Coulé,True,18.0,Guadalajara,6.0,0,11 trabajadores
15142,Falso,Touché-Coulé,NaN,17.0,Guadalajara,N.A.,1,14 Trabajadores
15143,NaN,Lobos,NaN,16.0,Zapopan,N.A.,1,15.5 Trabajadores
15144,NaN,Lobos,NaN,13.0,Guadalajara,N.A.,0,10 trabajadores
15145,Falso,Touché-Coulé,False,16.0,Guadalajara,4.0,1,11 Trabajadores
15146,Falso,Touché-Coulé,NaN,13.0,zapopan,N.A.,0,tenemos 10.5 trabajadores para los 7 dias de l...


In [ ]:
# 1. TYPE TRANSFORMATIONS

In [8]:
print("\n" + "=" * 60)
print("1. TYPE TRANSFORMATIONS")
print("=" * 60)
 
# ubicado_en_avenida: 'True'/'Falso' strings - bool
df["ubicado_en_avenida"] = df["ubicado_en_avenida"].map({"True": True, "Falso": False})
 
# abierto_en_fines_de_semana: object - boolean
df["abierto_en_fines_de_semana"] = df["abierto_en_fines_de_semana"].astype("boolean")
 
# dias_para_recoleccion_de_dinero: float64 - Int64 (no decimals)
df["dias_para_recoleccion_de_dinero"] = df["dias_para_recoleccion_de_dinero"].astype("Int64")
 
# cantidad_de_camaras: str with 'N.A.' - float (N.A. becomes NaN)
df["cantidad_de_camaras"] = pd.to_numeric(df["cantidad_de_camaras"], errors="coerce")
 
# banco_inseguro: int64 (0/1) - bool
df["banco_inseguro"] = df["banco_inseguro"].astype(bool)
 
print("New dtypes:")
print(df.dtypes)
 
print("\nNull values per column after transformation:")
print(df.isna().sum())


1. TYPE TRANSFORMATIONS
New dtypes:
ubicado_en_avenida                  object
empresa_recolectora_de_dinero          str
abierto_en_fines_de_semana         boolean
dias_para_recoleccion_de_dinero      Int64
municipio                              str
cantidad_de_camaras                float64
banco_inseguro                        bool
cantidad_de_empleados                  str
dtype: object

Null values per column after transformation:
ubicado_en_avenida                  2628
empresa_recolectora_de_dinero          0
abierto_en_fines_de_semana         10451
dias_para_recoleccion_de_dinero        0
municipio                              0
cantidad_de_camaras                10603
banco_inseguro                         0
cantidad_de_empleados                  0
dtype: int64


In [ ]:
# 2. TEXT STANDARDIZATION — municipio

In [9]:
print("\n" + "=" * 60)
print("2. TEXT STANDARDIZATION")
print("=" * 60)
 
print("Before standardization:")
print(df["municipio"].value_counts())
 
# Strip whitespace + capitalize
df["municipio"] = df["municipio"].str.strip().str.capitalize()
 
print("\nAfter standardization:")
vc = df["municipio"].value_counts()
print(vc)
 
second_most = vc.index[1]
second_count = vc.iloc[1]
print(f"\nSecond most frequent: '{second_most}' with {second_count} records")


2. TEXT STANDARDIZATION
Before standardization:
municipio
Guadalajara    10395
Zapopan         2531
zapopan         2184
Tlajomulco        20
Tlajomulco        17
Name: count, dtype: int64

After standardization:
municipio
Guadalajara    10395
Zapopan         4715
Tlajomulco        37
Name: count, dtype: int64

Second most frequent: 'Zapopan' with 4715 records
